In [4]:
import json
import pandas as pd
import Levenshtein
from copy import deepcopy

In [5]:
with open('/home/erik/Riksarkivet/Projects/fastighet/output/pipeline_flipside_backside_corr_with_gt_links2.json', 'r') as f:
    eval_dict = json.load(f)

In [3]:
df = pd.DataFrame(list(zip(fnr_list, trakt_list, block_list, enhet_list)), columns = ['FNR', 'GTRAKT', 'GBLOCK', 'GENHET'])

NameError: name 'fnr_list' is not defined

In [4]:
df_lm = pd.read_csv('/home/erik/Riksarkivet/Projects/fastighet/data/LM_2008.txt', sep=';', encoding='latin-1', dtype=str, index_col=[3,4,5,6])

In [6]:
# Skapa ett dict med batch och page som keys, sen trakt, block, enhet, fnr som keys

eval_series_dict = {}

trakt_lists = []
block_lists = []
enhet_lists = []
fnr_lists = []
trakt_gt_list = []
block_gt_list = []
enhet_gt_list = []
no_of_sock_mistakes = 0

i = 0
for batch in eval_dict:

    fnr_list = []
    trakt_list = []
    block_list = []
    enhet_list = []
    

    for page in eval_dict[batch].keys():
        if i % 2 == 0:
            if eval_dict[batch][page] is not None and eval_dict[batch][page][0]['det_prob'] is not None:
                link = False
                for inst in eval_dict[batch][page]:

                    if 'chosen' in inst.keys() and len(inst['pred_links']) == 1:
                        fnr_list.append(inst['pred_links'][0][1])
                        trakt_list.append(inst['pred_links'][0][0].split(';')[1])
                        block_list.append(inst['pred_links'][0][0].split(';')[2])
                        enhet_list.append(inst['pred_links'][0][0].split(';')[3])
                        link = True
                        

                    elif 'chosen' in inst.keys() and len(inst['sc_links']) == 1:
                        fnr_list.append(inst['sc_links'][0][1])
                        trakt_list.append(inst['sc_links'][0][0].split(';')[1])
                        block_list.append(inst['sc_links'][0][0].split(';')[2])
                        enhet_list.append(inst['sc_links'][0][0].split(';')[3])
                        link = True
                
                if not link:
                    fnr_list.append('0')
                    try:
                        trakt_list.append(inst['pred'].split(';')[0])
                        block_list.append(inst['pred'].split(';')[1])
                        enhet_list.append(inst['pred'].split(';')[2])
                    except:
                        trakt_list.append('0')
                        block_list.append('0')
                        enhet_list.append('0')

                    #count number of socken mistakes

            else:
                fnr_list.append('X')
                trakt_list.append('X')
                block_list.append('X')
                enhet_list.append('X')

        i += 1

    trakt_lists.append(trakt_list)
    block_lists.append(block_list)
    enhet_lists.append(enhet_list)
    fnr_lists.append(fnr_list)

0


In [ ]:
#add jump number to 

In [7]:
#if eval_dict[batch][page] is not None and eval_dict[batch][page][0]['det_prob'] is not None:
eval_dict_no = 0
lists = 0

i = 0

for batch in eval_dict:
    for page in eval_dict[batch].keys():
        if i % 2 == 0:   
            eval_dict_no += 1

        i += 1

for fnr_list in fnr_lists:
    for fnr in fnr_list:
        lists += 1

print(eval_dict_no)
print(lists)

59651
59651


In [34]:
count = 0
window_count = 0

for fnr_list in fnr_lists:

    ['0' if ]

171


In [3]:
a = [1,2,3,4,5,6]

a[1:3]

[2, 3]

In [12]:
total_num_of_ind_pages = 0
total_num_of_hits = 0
total_num_of_amb = 0
total_num_of_false_pos = 0


for batch, fnr_list in zip(eval_dict, fnr_lists):

    batch_num_of_ind_pages = 0
    batch_num_of_hits = 0
    batch_num_of_amb = 0
    batch_num_of_false_pos = 0
    pages = eval_dict[batch].keys()
    pages_odd = []
    for j, page in enumerate(pages):
        if j % 2 == 0:
            pages_odd.append(page)

    for ind, (page, fnr) in enumerate(zip(pages_odd, fnr_list)):

        if eval_dict[batch][page] is not None and eval_dict[batch][page][0]['det_prob'] is not None:
            total_num_of_ind_pages += 1
        
        try:
            gts_fnr = eval_dict[batch][page][0]['gts'][0][1]
            if fnr == gts_fnr:
                total_num_of_hits += 1

            elif fnr != gts_fnr and fnr != '0' and fnr != 'X':
                if abs(int(fnr) - int(fnr_list[ind-1])) > 50:
                    total_num_of_false_pos += 1
                    for k in range(ind + 1, len(fnr_list)):
                        if fnr_list[k] == fnr_list[ind]:
                            fnr_list[k] = '0'
                        else:
                            break
        except:
            pass

print((total_num_of_hits)/total_num_of_ind_pages)
print((total_num_of_false_pos)/total_num_of_ind_pages)



0.8306879217738603
0.024325161823440297


In [ ]:
total_num_of_ind_pages = 0
total_num_of_hits = 0
total_num_of_amb = 0
total_num_of_false_pos = 0
no_of_plus = 0

with open('/home/erik/Riksarkivet/Projects/fastighet/output/eval_series_window_size_3.txt', 'w') as f:
    f.write('BATCH' + '\t' + 'HIT_RATE' + '\t' + 'AMB_RATE' + '\t' + 'FP_RATE' + '\n')

    for batch in eval_dict.keys():

        batch_num_of_ind_pages = 0
        batch_num_of_hits = 0
        batch_num_of_amb = 0
        batch_num_of_false_pos = 0

        for page in eval_dict[batch].keys():

            if eval_dict[batch][page] is not None and eval_dict[batch][page][0]['det_prob'] is not None:

                total_num_of_ind_pages += 1
                batch_num_of_ind_pages += 1

                gts_pred = [x[0] for x in eval_dict[batch][page][0]['gts']]

                for gt in gts_pred:
                    if '+' in gt:
                        no_of_plus += 1
                        break

                for inst in eval_dict[batch][page]:
                    gts_pred = [x[0] for x in inst['gts']]
                    
                    gts_fnr = [x[1] for x in inst['gts']]

                    if 'chosen' in inst.keys():

                        false_pos = True
                        
                        if len(inst['pred_links']) > 1:
                            total_num_of_amb +=1
                            batch_num_of_amb +=1
                        elif len(inst['sc_links']) > 1:
                            total_num_of_amb +=1
                            batch_num_of_amb +=1

                        

                        for link in inst['pred_links']:
                            if link[1] in gts_fnr:
                                total_num_of_hits += 1
                                batch_num_of_hits += 1
                                false_pos = False
                                break
                        for link in inst['sc_links']:
                            false_pos = False
                            if link[1] in gts_fnr:
                                #total_num_of_hits += 1
                                #batch_num_of_hits += 1
                                
                                break

                        if false_pos:
                            total_num_of_false_pos += 1
                            batch_num_of_false_pos += 1

        hit_rate = str(batch_num_of_hits/batch_num_of_ind_pages)
        amb_rate = str(batch_num_of_amb/batch_num_of_ind_pages)
        fp_rate = str(batch_num_of_false_pos/batch_num_of_ind_pages)

        f.write(batch + '\t' + hit_rate + '\t' + amb_rate + '\t' + fp_rate + '\n')
        

print((total_num_of_hits)/total_num_of_ind_pages)
print(total_num_of_amb/total_num_of_ind_pages)
print((total_num_of_false_pos)/total_num_of_ind_pages)

In [41]:
df = pd.DataFrame(list(zip(fnr_list, trakt_list, block_list, enhet_list)), columns = ['FNR', 'GTRAKT', 'GBLOCK', 'GENHET'])